<a href="https://colab.research.google.com/github/JohriSumati-ops/CAR-PRICE-PREDICTOR-MODEL/blob/main/CAR_PRICE_PREDICTOR_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING

OBJECTIVE: Create a machine learning model based machine learning algorithm which uses parameters to determine the prices of different cars through categorical and numerical input data.

IMPORTING REQUIRED LIBRARIES

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


LOADING THE DATASET

In [3]:
df = pd.read_csv('/content/quikr_car (1).csv')
df.head()


,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


CLEANING THE DATA

In [5]:
# Drop rows where target is missing
df = df.dropna(subset=['Price'])

# Remove commas and convert Price to numeric if needed
df['Price'] = df['Price'].astype(str).str.replace(',', '')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

df = df.dropna(subset=['Price'])


SPLITING FEATURES AND TARGET

In [6]:
X = df.drop('Price', axis=1)
y = df['Price']


IDENTIFYING COLUMN TYPES

In [7]:
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(exclude=['object']).columns


PREPROCESSING AND LINEAR REGRESSION PIPELINE

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])


TRAIN TEST AND SPLIT

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


TRAIN THE MODEL

In [10]:
model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['name', 'company', 'year', 'kms_driven', 'fuel_type'], dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index([], dtype='object'))])),
                ('regressor', LinearRegression())])

PREDICTIONS AND EVALUATION

In [11]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Linear Regression Performance:")
print("R2 Score:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


Linear Regression Performance:
R2 Score: -1.846061786790857
MAE: 172722.04348509715
RMSE: 627141.325663898


TESTING

In [12]:
# Example (change values based on your dataset columns)
sample = X.iloc[[0]]
predicted_price = model.predict(sample)

print("Predicted Price:", predicted_price[0])


Predicted Price: 80000.36135652912


MODEL AS A WEBSITE

In [1]:
!pip install streamlit pyngrok scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 15.0 MB/s eta 0:00:00


In [14]:
import pickle
# Sample dataset (replace with your Kaggle dataset if needed)
data = {
    'year': [2015, 2016, 2017, 2018, 2019],
    'km_driven': [50000, 40000, 30000, 20000, 10000],
    'fuel': [1, 1, 0, 0, 1],   # 1=Petrol, 0=Diesel
    'price': [5, 5.5, 6, 6.5, 7]  # in lakhs
}

df = pd.DataFrame(data)

X = df[['year', 'km_driven', 'fuel']]
y = df['price']

model = LinearRegression()
model.fit(X, y)

pickle.dump(model, open('car_price_model.pkl', 'wb'))


In [15]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np

# Load model
model = pickle.load(open('car_price_model.pkl', 'rb'))

st.set_page_config(page_title="Car Price Predictor", layout="centered")

st.title("🚗 Car Price Predictor")
st.write("Predict the price of your car using Machine Learning")

year = st.number_input("Car Manufacturing Year", 2000, 2024, 2018)
km = st.number_input("Kilometers Driven", 0, 300000, 40000)
fuel = st.selectbox("Fuel Type", ["Petrol", "Diesel"])

fuel_val = 1 if fuel == "Petrol" else 0

if st.button("Predict Price"):
    features = np.array([[year, km, fuel_val]])
    prediction = model.predict(features)
    st.success(f"Estimated Car Price: ₹ {round(prediction[0], 2)} Lakhs")


Writing app.py


In [16]:
from pyngrok import ngrok

ngrok.set_auth_token("38VW8XITRMWMDbCkBHKwZixLWsK_78MESc47PJXX2XXZXXcgV")

!streamlit run app.py &>/dev/null&

public_url = ngrok.connect(8501)
public_url


<NgrokTunnel: "https://unlured-tianna-northernmost.ngrok-free.dev" -> "http://localhost:8501">